# Train
Run this notebook end-to-end to train the SpectralTransformer.
**Step 1:** Run the install + inspection cells. **Step 2:** Run tiny training to confirm the pipeline. **Step 3:** Run full training.

In [ ]:
%pip install -q torch datasets tqdm numpy

In [ ]:
import os, sys, math
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import config

In [ ]:
%run data.ipynb
%run model.ipynb

In [ ]:
def setup_environment():
    """Mount Google Drive when running in Colab; fall back to local checkpoints directory.

    Returns:
        str: Absolute path to the checkpoint directory.
    """
    in_colab = 'google.colab' in sys.modules
    if in_colab:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        checkpoint_dir = '/content/drive/MyDrive/RAST/checkpoints'
        print('Running in Colab — checkpoints saving to Google Drive.')
    else:
        checkpoint_dir = os.path.abspath(os.path.join(os.getcwd(), '..', config.CHECKPOINT_DIR))
        print('Running locally — checkpoints saving to checkpoints/.')
    os.makedirs(checkpoint_dir, exist_ok=True)
    return checkpoint_dir

In [ ]:
def get_scheduler(optimizer):
    """Build a scheduler with linear warmup followed by cosine annealing.

    Args:
        optimizer: AdamW optimizer instance.

    Returns:
        torch.optim.lr_scheduler.LambdaLR
    """
    def lr_lambda(epoch):
        if epoch < config.WARMUP_EPOCHS:
            return (epoch + 1) / config.WARMUP_EPOCHS
        progress = (epoch - config.WARMUP_EPOCHS) / max(1, config.EPOCHS - config.WARMUP_EPOCHS)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [ ]:
def joint_loss(recon_pred, recon_target, z_pred, z_target, lambda_z=config.LAMBDA_REDSHIFT):
    """Compute the joint reconstruction and redshift prediction loss.

    Args:
        recon_pred (torch.Tensor): Predicted patches at masked positions, shape (B, n_masked, PATCH_SIZE).
        recon_target (torch.Tensor): Ground-truth patches at masked positions, shape (B, n_masked, PATCH_SIZE).
        z_pred (torch.Tensor): Predicted redshift values, shape (B, 1).
        z_target (torch.Tensor): Ground-truth redshift values, shape (B, 1).
        lambda_z (float): Weighting factor for the redshift loss term.

    Returns:
        tuple: (total_loss, L_recon, lambda_z * L_z) — all scalar tensors.
    """
    L_recon = F.mse_loss(recon_pred, recon_target)
    L_z_weighted = lambda_z * F.mse_loss(z_pred, z_target)
    return L_recon + L_z_weighted, L_recon, L_z_weighted

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    """Run one full pass over the training set.

    Args:
        model (SpectralTransformer): The model being trained.
        loader (DataLoader): Training data loader.
        optimizer: Optimizer instance.
        device (torch.device): Target compute device.

    Returns:
        tuple: (mean_total_loss, mean_recon_loss, mean_z_loss) over the epoch.
    """
    model.train()
    total, recon_sum, z_sum = 0.0, 0.0, 0.0

    for flux, redshift in tqdm(loader, desc='Train', leave=False):
        flux     = flux.to(device)
        redshift = redshift.unsqueeze(1).to(device)
        B = flux.shape[0]

        mask = random_mask(B, config.N_PATCHES).to(device)
        patches = flux.reshape(B, config.N_PATCHES, config.PATCH_SIZE)
        n_masked = mask.sum(dim=1)[0].item()
        target_patches = patches[mask].reshape(B, int(n_masked), config.PATCH_SIZE)

        recon_pred, z_pred = model(flux, mask)
        loss, L_recon, L_z = joint_loss(recon_pred, target_patches, z_pred, redshift)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total     += loss.item()
        recon_sum += L_recon.item()
        z_sum     += L_z.item()

    n = len(loader)
    return total / n, recon_sum / n, z_sum / n


def validate(model, loader, device):
    """Evaluate the model on the validation set without gradient updates.

    Args:
        model (SpectralTransformer): The model to evaluate.
        loader (DataLoader): Validation data loader.
        device (torch.device): Target compute device.

    Returns:
        tuple: (mean_total_loss, mean_recon_loss, mean_z_loss) over the epoch.
    """
    model.eval()
    total, recon_sum, z_sum = 0.0, 0.0, 0.0

    with torch.no_grad():
        for flux, redshift in tqdm(loader, desc='Val', leave=False):
            flux     = flux.to(device)
            redshift = redshift.unsqueeze(1).to(device)
            B = flux.shape[0]

            mask = random_mask(B, config.N_PATCHES).to(device)
            patches = flux.reshape(B, config.N_PATCHES, config.PATCH_SIZE)
            n_masked = mask.sum(dim=1)[0].item()
            target_patches = patches[mask].reshape(B, int(n_masked), config.PATCH_SIZE)

            recon_pred, z_pred = model(flux, mask)
            loss, L_recon, L_z = joint_loss(recon_pred, target_patches, z_pred, redshift)

            total     += loss.item()
            recon_sum += L_recon.item()
            z_sum     += L_z.item()

    n = len(loader)
    return total / n, recon_sum / n, z_sum / n

## Step 1 — Tiny training run
Confirm the full pipeline works before committing to full training.
Loss should decrease and neither recon nor z*5 should be NaN.

In [ ]:
torch.manual_seed(42)
checkpoint_dir = setup_environment()
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Device: {device}')

tiny_train, tiny_val = get_dataloaders(batch_size=4, n_train=16, n_val=8)
tiny_model = SpectralTransformer().to(device)
tiny_opt   = AdamW(tiny_model.parameters(), lr=config.LR, weight_decay=1e-4)

for epoch in range(3):
    tl, tr, tz = train_one_epoch(tiny_model, tiny_train, tiny_opt, device)
    vl, vr, vz = validate(tiny_model, tiny_val, device)
    print(
        f'Epoch {epoch+1}/3 | '
        f'train  total={tl:.4f}  recon={tr:.4f}  z*5={tz:.4f} | '
        f'val    total={vl:.4f}  recon={vr:.4f}  z*5={vz:.4f}'
    )

print('Pipeline check passed — proceed to full training.')

## Step 2 — Full training
Only run this after the tiny training cell confirms the pipeline is working.

In [ ]:
torch.manual_seed(42)

train_loader, val_loader = get_dataloaders()
model = SpectralTransformer().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

optimizer = AdamW(model.parameters(), lr=config.LR, weight_decay=1e-4)
scheduler = get_scheduler(optimizer)
best_val_loss = float('inf')

for epoch in range(config.EPOCHS):
    tl, tr, tz = train_one_epoch(model, train_loader, optimizer, device)
    vl, vr, vz = validate(model, val_loader, device)
    scheduler.step()

    lr_now = scheduler.get_last_lr()[0]
    print(
        f'Epoch {epoch+1:3d}/{config.EPOCHS} | '
        f'train  total={tl:.4f}  recon={tr:.4f}  z*5={tz:.4f} | '
        f'val    total={vl:.4f}  recon={vr:.4f}  z*5={vz:.4f} | '
        f'lr={lr_now:.2e}'
    )

    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), os.path.join(checkpoint_dir, 'best.pt'))
        print(f'  -> Saved best checkpoint')